# 13 · Evaluation of Fine-Tuned Models

In plain English, this notebook is about answering one simple-sounding question: **"Is my fine-tuned model actually any good?"** You just spent earlier notebooks learning how to *train* a model. But a model that trains without errors is not the same as a model that *works*. To know whether it's good, you have to measure it — and measure it the *right* way, because the most obvious measurement (plain accuracy) can fool you badly.

We'll use the same setting as the course capstone all the way through: a **lead-intent classifier** that labels each sales lead as **`hot`** (ready to buy), **`warm`** (interested), or **`cold`** (not interested). Every metric we learn here is exactly what you'll report in notebook 15 after fine-tuning that classifier.

Everything in this notebook runs on a plain laptop CPU in seconds — we use small, hand-crafted prediction arrays so you can focus on *understanding the metrics*, not on training.

## What you'll learn

- **Why accuracy alone can mislead you** — the classic "imbalanced classes" trap.
- **Accuracy** — `accuracy_score`: the fraction of predictions that were correct.
- **Precision** and **Recall** — what they mean *in plain words* for hot/warm/cold leads, and the trade-off between them.
- **F1 score** — a single number that balances precision and recall, plus **macro vs weighted** averaging.
- **`classification_report`** — the one function that prints all of the above as a neat per-class table.
- **Confusion matrix** — `confusion_matrix` plus a matplotlib heatmap, and how to *read* it to see which classes get mixed up.
- **Manual review** — why you should always eyeball real examples by hand, not just trust the numbers.
- **Baseline vs fine-tuned** — comparing a dumb "always guess the most common class" baseline to your model, so you can prove it actually improved.

## Why this matters for fine-tuning

Fine-tuning is an experiment, and every experiment needs a way to tell success from failure. Without solid evaluation you can't answer the questions that actually matter:

- Did fine-tuning **help**, or did the model get worse?
- Is the model good at **all three** lead types, or only the common one?
- Where does it make mistakes — does it call **cold** leads **hot** (wasting your sales team's time) or **hot** leads **cold** (missing real deals)?
- Is my new model better than a **dumb baseline**? If not, fine-tuning bought me nothing.

You can only improve what you can measure. The metrics in this notebook are the dashboard you'll watch while tuning — and the exact numbers you'll report for your capstone model in notebook 15.

## Setup

Run the cell below once. The `%pip install` line is **commented out** — uncomment it if you're on Google Colab or a fresh environment. These libraries are tiny and CPU-only.

In [ ]:
# Uncomment the next line on Colab or a fresh environment:
# %pip install scikit-learn matplotlib numpy

import numpy as np                 # arrays and simple number-crunching
import matplotlib.pyplot as plt    # for the confusion-matrix heatmap

# scikit-learn gives us every evaluation metric we need, ready-made:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

print("Imports OK — ready to evaluate.")

## 1. Our evaluation data: true labels vs predicted labels

To evaluate *any* classifier, you only ever need two things:

- **`y_true`** — the **correct** answer for each example (the "ground truth", what a human labeled it).
- **`y_pred`** — what the **model guessed** for each example.

Every metric in this notebook is just a different way of comparing these two lists. You do **not** need the trained model itself to compute them — only its predictions on a test set it has never seen.

Below we hand-craft a realistic test set of 40 leads. Notice it's **imbalanced**: most leads are `cold` (that's true in real sales data — most people aren't ready to buy). The predictions are *mostly* right but contain realistic mistakes, so our metrics will be interesting rather than perfect.

In [ ]:
# The TRUE labels for 40 test leads (what a human reviewer decided).
# This is deliberately imbalanced: lots of "cold", fewer "warm", fewest "hot".
y_true = [
    "cold","cold","cold","cold","cold","cold","cold","cold","cold","cold",
    "cold","cold","cold","cold","cold","cold","cold","cold","cold","cold",
    "cold","cold","cold","cold","warm","warm","warm","warm","warm","warm",
    "warm","warm","warm","warm","hot","hot","hot","hot","hot","hot",
]

# What our (pretend) fine-tuned model PREDICTED for those same 40 leads.
# Most are correct, but we baked in believable errors:
#  - a few cold leads wrongly called warm
#  - a few warm leads called cold (missed) or hot (over-eager)
#  - one hot lead missed as warm
y_pred = [
    "cold","cold","cold","cold","cold","cold","cold","cold","cold","cold",
    "cold","cold","cold","cold","cold","cold","cold","cold","cold","cold",
    "cold","warm","warm","cold","warm","warm","warm","warm","warm","cold",
    "cold","warm","hot","warm","hot","hot","hot","hot","warm","hot",
]

# The fixed set of class names, in a consistent order we'll reuse everywhere.
labels = ["cold", "warm", "hot"]

print("Number of test leads:", len(y_true))
print("How many of each TRUE label:")
for name in labels:
    print(f"  {name:5s}: {y_true.count(name)}")
# Expected: cold 24, warm 10, hot 6  -> clearly imbalanced toward "cold".

**What this does:**

- `y_true` and `y_pred` are plain Python lists of strings — scikit-learn happily works with text labels, no need to convert to numbers.
- We define `labels = ["cold", "warm", "hot"]` once and pass it to every metric. Fixing the order matters so the confusion-matrix rows/columns line up the way we expect.
- The counts confirm the **imbalance**: 24 cold, 10 warm, 6 hot. That imbalance is the whole reason the next section matters.

### ✏️ Exercise

Print the count of each label in **`y_pred`** (use `.count()` like we did for `y_true`). Compare to the true counts: does the model predict "cold" more often or less often than it should? Jot down what you notice — we'll confirm it with the confusion matrix later.

## 2. Why accuracy alone can mislead you

**Intuition first.** Accuracy = "what fraction of predictions were correct?" It sounds like the perfect score, but it has a famous trap: **imbalanced classes**.

Our data is 24/40 = **60% cold**. Imagine a lazy model that ignores every lead and *always shouts "cold!"*. It would be right on all 24 cold leads and wrong on the 16 others — scoring **60% accuracy** while being completely useless: it never finds a single hot lead, so your sales team gets nothing actionable.

In a more extreme business dataset where 80% of leads are cold, a "always cold" model scores **80% accuracy** and is *still* worthless. High accuracy can hide total failure on the classes you actually care about (the rare `hot` leads that make you money). Let's prove it.

In [ ]:
# The "lazy" model: predict the most common class ("cold") for EVERY lead.
always_cold = ["cold"] * len(y_true)

lazy_acc = accuracy_score(y_true, always_cold)
print(f"Accuracy of the 'always cold' model: {lazy_acc:.0%}")
# Expected: 60% -- just because 60% of the leads really are cold.

# But how many HOT leads did it catch? Let's count.
hot_caught = sum(1 for t, p in zip(y_true, always_cold) if t == "hot" and p == "hot")
print(f"Hot leads correctly identified by the lazy model: {hot_caught} out of {y_true.count('hot')}")
# Expected: 0 out of 6 -- it never finds a hot lead. Useless, despite 60% accuracy.

**What this does:**

- `["cold"] * len(y_true)` builds a prediction list that is just "cold" 40 times — our lazy baseline.
- `accuracy_score` shows it scores a respectable-looking **60%**...
- ...but counting the hot leads it caught reveals the truth: **zero**. The lesson: **never judge an imbalanced classifier by accuracy alone.** You need per-class metrics, which is exactly what precision and recall give us next.

### ✏️ Exercise

Build an `always_warm = ["warm"] * len(y_true)` model and print its accuracy. Why is it *lower* than the "always cold" model's accuracy? (Hint: how many leads are truly `warm` vs truly `cold`?)

## 3. Accuracy — the honest version

Accuracy still has a place: it's the simplest one-number summary, and for our *real* model (not the lazy one) it's a fine starting point. Just never let it be your *only* number.

`accuracy_score(y_true, y_pred)` returns a value between 0 and 1 = (number correct) / (total).

In [ ]:
acc = accuracy_score(y_true, y_pred)
print(f"Accuracy of our model: {acc:.2%}")

# Let's verify it by hand to demystify the function:
correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
print(f"Correct predictions: {correct} / {len(y_true)} = {correct/len(y_true):.2%}")
# The two numbers match -- accuracy_score is literally "correct / total".

**What this does:**

- `accuracy_score` compares the two lists element-by-element and returns the fraction that match.
- The hand-computed `correct / total` matches it exactly — there's no magic here. Our model lands around **80%**, clearly better than the 60% lazy baseline, but accuracy doesn't tell us *where* the 20% of errors are. That's the job of precision and recall.

## 4. Precision and Recall — the two questions that really matter

This is the most important section in the notebook. **Read the intuition slowly** — precision and recall are the metrics professionals actually argue about.

Pick one class to focus on, say **`hot`**. There are two completely different ways to be "good" at hot leads, and they answer two different business questions:

- **Precision of `hot`** = *Of all the leads we LABELED hot, how many were really hot?*
  This protects your sales team's **time**. Low precision means you keep telling salespeople "this lead is hot, call them now!" and they waste effort on people who weren't actually ready. Precision = "**when I say hot, can you trust me?**"

- **Recall of `hot`** = *Of all the leads that were truly hot, how many did we CATCH?*
  This protects your **revenue**. Low recall means real, ready-to-buy customers slipped through labeled as warm or cold, and you **missed the deal**. Recall = "**of the real hot leads out there, how many did I find?**"

The formulas, in words (no scary math):

- Precision = correct hot calls ÷ **all the times we said hot**.
- Recall = correct hot calls ÷ **all the leads that were actually hot**.

Let's compute both, per class.

In [ ]:
# precision_score / recall_score need to know which classes exist (labels=)
# and how to combine the per-class numbers. average=None means:
#   "don't combine -- give me one number PER class, in `labels` order."
prec_per_class = precision_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
rec_per_class  = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)

print("Per-class precision and recall:")
print(f"{'class':6s} {'precision':>10s} {'recall':>8s}")
for name, p, r in zip(labels, prec_per_class, rec_per_class):
    print(f"{name:6s} {p:10.2f} {r:8.2f}")

**What this does:**

- `average=None` tells scikit-learn: *don't blend the classes together* — return one precision and one recall for **each** of cold/warm/hot.
- `labels=labels` fixes the order so the rows line up with our `["cold","warm","hot"]`.
- `zero_division=0` just says "if a class was never predicted, report 0 instead of crashing with a warning."

**How to read the `hot` row:** its **precision** tells you how often "hot" was correct *when the model said hot*; its **recall** tells you how many of the truly-hot leads it managed to catch. Look at where precision and recall *differ* for a class — that gap tells you whether the model is over-eager (high recall, low precision) or too cautious (high precision, low recall) about that class.

### The precision/recall trade-off

You usually **can't max out both at once**, and you get to choose which to favor based on the business cost of each mistake:

- **Favor recall for `hot`** if missing a ready buyer is expensive — you'd rather flag a few extra leads as hot (and waste a little sales time) than let a real deal slip away. You'll catch more hot leads but make more false alarms (precision drops).
- **Favor precision for `hot`** if your sales team is small and their time is precious — you only want to cry "hot!" when you're sure. You'll waste less time but miss some real hot leads (recall drops).

There's no universally "right" choice — it depends on what a mistake *costs your business*. F1 (next) is for when you want a single number that balances the two.

### ✏️ Exercise

Using the table you just printed, answer in a comment: for the **`warm`** class, is the model better at **precision** or **recall**? In plain words, does it tend to *miss* warm leads, or *over-call* leads as warm? (Peek at the confusion matrix in section 7 if you want to confirm your answer.)

## 5. F1 score — balancing precision and recall in one number

**Intuition first.** Precision and recall pull in opposite directions, so people want *one* number that rewards being good at **both**. That's the **F1 score**.

F1 is the "harmonic mean" of precision and recall — you don't need that math; just know its key property: **F1 is only high when precision AND recall are both high.** If either one is low, F1 is dragged down. So you can't game it by, say, predicting "hot" for everything (great recall, terrible precision → mediocre F1).

When you have multiple classes, you need to combine the per-class F1 scores into one. There are two common ways:

- **Macro average** — a *plain average* of the three class F1 scores. Every class counts **equally**, no matter how rare. This is the one to watch when the **rare `hot` class matters as much as the common `cold` class** (it usually does in sales!).
- **Weighted average** — averages the class F1 scores but **weights each by how many examples it has**. Common classes dominate. This tracks "overall" performance but can hide a terrible score on the rare class.

Rule of thumb for imbalanced data like ours: **report macro F1** so the rare-but-valuable `hot` class can't be swept under the rug.

In [ ]:
f1_per_class = f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
print("Per-class F1:")
for name, f in zip(labels, f1_per_class):
    print(f"  {name:5s}: {f:.2f}")

# Now the two ways of combining them into one number:
f1_macro    = f1_score(y_true, y_pred, labels=labels, average="macro",    zero_division=0)
f1_weighted = f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)

print(f"\nMacro    F1 (every class counts equally): {f1_macro:.2f}")
print(f"Weighted F1 (common classes count more): {f1_weighted:.2f}")
# Weighted is usually HIGHER here, because the model does well on the common
# "cold" class, which the weighted average rewards heavily.

**What this does:**

- `average=None` gives the F1 for each class; you can see immediately which class the model struggles with most (the lowest F1).
- `average="macro"` is the plain mean of those three — it **drops** if any single class is weak, even a rare one.
- `average="weighted"` leans toward the common `cold` class, so it usually looks rosier.
- The gap between macro and weighted is a **red flag for imbalance**: if weighted is much higher than macro, your model is coasting on the easy majority class and probably weak on the rare ones. Watch macro F1 to keep yourself honest.

### ✏️ Exercise

Compute and print **`precision_score(..., average="macro")`** and **`recall_score(..., average="macro")`**. Then check by hand that macro F1 sits *between* macro precision and macro recall — that's what "balancing the two" looks like.

## 6. `classification_report` — all of it in one table

Computing each metric separately is great for understanding, but in practice you'll reach for **one** function that prints everything at once: `classification_report`. It gives you precision, recall, F1, and the **support** (how many true examples each class had) per class, plus the macro and weighted averages — the whole dashboard in a single call.

This is the table you'll paste into your capstone results in notebook 15.

In [ ]:
report = classification_report(y_true, y_pred, labels=labels, zero_division=0)
print(report)

**What this does — how to read the table together:**

- Each **row** (cold / warm / hot) shows that class's **precision**, **recall**, **f1-score**, and **support** (number of true examples — notice cold=24, warm=10, hot=6, matching our imbalance).
- **`accuracy`** appears once near the bottom — it's the overall fraction correct (the same number from section 3).
- **`macro avg`** = the plain average across classes (every class equal).
- **`weighted avg`** = average weighted by support (big classes dominate).

**Reading tip:** scan *down* the `recall` column to spot classes the model **misses**, and *down* the `precision` column to spot classes it **over-calls**. The `hot` row is the one to watch in a sales setting — low hot-recall means lost deals. The `support` column reminds you how much to trust each row: a class with support of only 6 is measured on very few examples, so treat its score as noisier.

### ✏️ Exercise

Add `output_dict=True` to the call: `rep = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)`. Now `rep` is a normal Python dictionary. Print `rep["hot"]["recall"]` to pull out a single metric programmatically — handy when you want to log one number across many fine-tuning runs.

## 7. Confusion matrix — *where* the mistakes happen

Metrics tell you *how much* the model is wrong; the **confusion matrix** tells you *how* it's wrong — which classes get mixed up with which. It's a grid:

- Each **row** = the **true** class.
- Each **column** = the **predicted** class.
- Each cell = how many leads had that (true, predicted) combination.

So the cell at row `hot`, column `warm` = "how many truly-hot leads the model wrongly called warm" (missed deals!). The **diagonal** (top-left to bottom-right) is where true == predicted — those are the **correct** ones. Everything **off the diagonal** is a mistake, and its position tells you the *type* of mistake.

Let's compute the raw numbers first.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=labels)
print("Confusion matrix (rows = TRUE, columns = PREDICTED):")
print("labels order:", labels)
print(cm)
# Reading example: cm[2] is the "hot" row. cm[2][1] = truly-hot leads
# that were predicted "warm" (a missed hot lead).

**What this does:**

- `confusion_matrix(y_true, y_pred, labels=labels)` returns a 3×3 grid of counts in our `["cold","warm","hot"]` order.
- Reading it raw is doable but tedious. A **heatmap** makes the pattern jump out instantly — bright diagonal = good, bright off-diagonal cells = the specific confusions to fix. Let's plot it.

In [ ]:
# ConfusionMatrixDisplay draws the grid with numbers and a color scale for us.
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, cmap="Blues", colorbar=False)   # darker cell = more leads
ax.set_title("Lead-intent confusion matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
plt.tight_layout()
plt.show()
# You should see a strong dark diagonal (correct predictions) with a few
# lighter off-diagonal cells showing where leads were confused.

**What this does:**

- `ConfusionMatrixDisplay(...).plot(...)` turns the count grid into a labeled heatmap; the number is printed in each cell and the color shows magnitude (darker = more).
- **How to read it:** follow the **diagonal** for correct predictions. Then look for the **brightest off-diagonal cell** — that's the model's most common mistake. In our data you'll see some `cold`↔`warm` confusion (the boundary between "not interested" and "mildly interested" is genuinely fuzzy) and a hot lead or two leaking into `warm`. That single picture tells you *exactly* which classes to focus on improving.

### ✏️ Exercise

Re-plot the confusion matrix with **`normalize="true"`** by passing it to `ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=labels, normalize="true", cmap="Blues")`. Now each **row sums to 1.0**, so cells show *percentages within each true class*. Why is this fairer than raw counts when the classes are imbalanced? (Hint: the `cold` row has 24 leads and the `hot` row only 6.)

## 8. Manual review — always eyeball the actual examples

**Numbers can lie by omission.** A confusion matrix tells you *that* 3 cold leads became warm, but not *why*. The only way to understand the *why* — and to catch silent bugs like mislabeled data or a broken prompt — is to **read the actual mistakes by hand**. Professionals do this on every project, every time. It's the cheapest, highest-value habit in all of evaluation.

Let's attach short text descriptions to our leads and build a small review table of (lead text, true label, predicted label), then filter it down to just the mistakes and read them.

In [ ]:
# A short text description for each of the 40 leads (parallel to y_true/y_pred).
# In a real project these come straight from your dataset's lead descriptions.
lead_texts = [
    "Browsed pricing once, no contact",        "Unsubscribed from emails",
    "Opened newsletter, no clicks",            "Visited blog, bounced fast",
    "Old lead, no activity in months",         "Free trial expired, never returned",
    "Clicked ad, left immediately",            "Generic info request, went quiet",
    "Downloaded ebook, no follow-up",          "Followed on social, nothing since",
    "Cold list import, never engaged",         "Attended webinar, no questions",
    "Visited careers page (not a buyer)",      "Competitor research, low intent",
    "One pageview last quarter",               "Bounced from signup form",
    "Asked for unsubscribe link",              "Read one help article",
    "Student, no budget mentioned",            "Looking for free tools only",
    "Viewed homepage twice, slow",             "Compared plans, no demo yet",
    "Asked one pricing question, paused",      "Signed up for tips, lukewarm",
    "Requested a callback next month",         "Downloaded buyer's guide",
    "Asked about team pricing",                "Compared us to a competitor",
    "Returned to pricing 3 times",             "Started a trial, light usage",
    "Mentioned budget, no timeline",           "Asked for case studies",
    "Booked then rescheduled a demo",          "Engaged sales over email twice",
    "Requested a quote, urgent timeline",      "Booked a demo for this week",
    "Asked to start onboarding ASAP",          "Ready to sign, needs contract",
    "Trial power user, asked to buy",          "Confirmed budget and decision date",
]

assert len(lead_texts) == len(y_true) == len(y_pred)  # safety: all aligned
print("Lead texts attached:", len(lead_texts))

**What this does:**

- `lead_texts` is a list of short, human-readable descriptions, **lined up index-for-index** with `y_true` and `y_pred` (lead `i` has text `lead_texts[i]`, truth `y_true[i]`, prediction `y_pred[i]`).
- The `assert` is a cheap safety check that all three lists are the same length — misaligned lists are a classic, silent evaluation bug.

In [ ]:
# Build a small review table and print ONLY the mistakes (true != predicted).
print(f"{'#':>2}  {'TRUE':5s} {'PRED':5s}  lead description")
print("-" * 60)
mistakes = 0
for i, (text, t, p) in enumerate(zip(lead_texts, y_true, y_pred)):
    if t != p:                      # only show the errors
        mistakes += 1
        flag = "  <-- MISSED HOT" if t == "hot" else ""
        print(f"{i:>2}  {t:5s} {p:5s}  {text}{flag}")
print("-" * 60)
print(f"Total mistakes to review: {mistakes} out of {len(y_true)}")

**What this does:**

- We loop over the three aligned lists at once with `zip`, and print a row **only when `t != p`** (a mistake).
- Reading these rows is where insight happens. For example, a lead described "Requested a callback next month" being predicted `cold` when it's truly `warm` is *understandable* (the phrasing is lukewarm) — maybe even arguably correct, which makes you question the original label. That kind of judgment **never** shows up in a metric; you only get it by reading.
- The `<-- MISSED HOT` flag highlights the most expensive error type (a truly hot lead the model failed to flag). In sales, those are the rows you'd escalate first.

### ✏️ Exercise

Change the loop to print only the **most costly** mistakes: leads where `t == "hot"` but `p != "hot"` (real deals the model missed). How many are there, and do their descriptions share anything in common that might explain the miss?

## 9. Baseline vs fine-tuned — proving the model actually helped

The single most important comparison in any ML project: **is my model better than doing something trivial?** If a fine-tuned model can't beat a dumb baseline, the fine-tuning was a waste.

The simplest baseline is the **majority-class** model from section 2: always predict the most common label (`cold`). It's the bar your real model must clear. Let's put the baseline and the model side by side on the metrics that matter for imbalanced data — **accuracy** and, crucially, **macro F1**.

In [ ]:
baseline_pred = ["cold"] * len(y_true)     # the majority-class baseline

def summarize(name, y_t, y_p):
    acc   = accuracy_score(y_t, y_p)
    macro = f1_score(y_t, y_p, labels=labels, average="macro", zero_division=0)
    hot_recall = recall_score(y_t, y_p, labels=["hot"], average="macro", zero_division=0)
    print(f"{name:22s}  accuracy={acc:5.2%}  macro-F1={macro:4.2f}  hot-recall={hot_recall:4.2f}")

print(f"{'model':22s}  {'metrics':s}")
print("-" * 60)
summarize("Majority baseline",  y_true, baseline_pred)
summarize("Fine-tuned model",   y_true, y_pred)

**What this does:**

- `summarize` prints three numbers for any set of predictions: overall **accuracy**, **macro F1** (the imbalance-aware headline metric), and the **recall of `hot`** specifically (the business-critical one).
- Watch what happens: the baseline's **accuracy looks okay-ish (60%)**, but its **macro F1 is terrible** and its **hot-recall is 0** — it never finds a hot lead. The fine-tuned model wins clearly on macro F1 and hot-recall. *That* is how you prove fine-tuning earned its keep.
- **Lesson:** always report your model **next to a baseline**, and judge the win on a metric that respects the rare classes (macro F1), not just accuracy.

### ✏️ Exercise

Add a third, slightly smarter baseline: predict `"warm"` for everything, or write a one-line rule like *"predict `hot` if the lead text contains the word 'demo' or 'quote', else `cold`"* and run it through `summarize`. Does your hand-written rule beat the majority baseline on macro F1? This is exactly the kind of baseline you'll compare your capstone model against.

## Common mistakes & how to debug them

- **Trusting accuracy on imbalanced data.** If one class dominates, accuracy can be high while the model is useless on the classes you care about. Always also report **per-class precision/recall** and **macro F1**.
- **Reporting only the weighted average.** Weighted average flatters models that ace the common class. If weighted F1 is much higher than **macro** F1, your rare classes are probably weak — look closer.
- **`y_true` and `y_pred` out of order or different lengths.** Every metric assumes example `i` lines up across both lists. A misalignment gives wrong-but-not-crashing numbers. Keep them paired (zip them, and `assert` equal lengths).
- **Confusing rows and columns in the confusion matrix.** **Rows = true, columns = predicted.** Reading them backwards flips precision and recall in your head. Label your axes every time.
- **Forgetting `labels=` (or using an inconsistent order).** Without a fixed `labels` list, scikit-learn picks alphabetical order, so your matrix rows may not match your mental model. Define `labels` once and pass it everywhere.
- **A `zero_division` warning.** If a class is never predicted, precision for it is 0/0. Pass `zero_division=0` to report 0 cleanly instead of getting a warning — but also *notice* it: a class the model never predicts is a real problem.
- **Skipping manual review.** Numbers can hide mislabeled data, a broken prompt, or "mistakes" that are actually correct. Always read a handful of real errors by hand.
- **Evaluating on the training data.** Metrics only mean something on a **held-out** set the model never saw during fine-tuning. Reporting training-set scores massively overstates how good the model is.

## Summary

- Evaluation = comparing **`y_true`** (ground truth) with **`y_pred`** (model guesses). You don't need the model itself, just its predictions on a held-out set.
- **Accuracy** (`accuracy_score`) is a simple one-number summary but **misleads on imbalanced data** — a lazy "always cold" model scored 60% while catching zero hot leads.
- **Precision** of a class = "when I say this class, am I right?" (protects time). **Recall** = "of the real ones, how many did I catch?" (protects revenue). They **trade off**, and which to favor depends on what a mistake costs.
- **F1** balances precision and recall into one number; use **macro** averaging on imbalanced data so the rare-but-valuable `hot` class isn't ignored. **Weighted** averaging flatters the common class.
- **`classification_report`** prints precision/recall/F1/support per class plus the averages — your one-stop results table.
- The **confusion matrix** (rows = true, columns = predicted) shows *where* mistakes happen; the heatmap makes the worst confusions obvious at a glance.
- **Manual review** of real errors catches what metrics can't — always do it.
- Always compare your model to a **baseline** (majority class) on an imbalance-aware metric like **macro F1** to prove fine-tuning actually helped.

These are the **exact metrics you'll report in the capstone (notebook 15)** after fine-tuning your lead-intent classifier.

## What to learn next

Next up: **`14_inference_and_serving.ipynb`**. You can now *measure* whether a fine-tuned model is good — the next notebook is about *using* it for real: running generation/inference cleanly, merging LoRA adapters back into the base model, saving and loading the finished model, and serving it behind a simple API so an application (or your sales team's tool) can call it. Then notebook 15 ties training, evaluation, and inference together in the capstone.